In [1]:
import pandas as pd
import numpy as np
from itertools import permutations
from statsmodels.tsa.stattools import coint

Hypothesis: GOOG and GOOGL are cointegrated, because they're the same, and can be used for pairs trading

In [2]:
filepath = "/Users/benjaminsumacbookm4/Desktop/Pairs Trading/data/market_data/IntradayMarketDataGOOG_GOOGL"
data = pd.read_parquet(filepath)

In [3]:
data.head()

,symbol,timestamp,close
0,GOOGL,2020-11-23 12:30:00+00:00,7.461640
1,GOOGL,2020-11-23 12:47:00+00:00,7.461640
2,GOOGL,2020-11-23 12:55:00+00:00,7.461640
3,GOOGL,2020-11-23 13:00:00+00:00,7.460491
4,GOOGL,2020-11-23 13:05:00+00:00,7.461767


In [4]:
data.tail()

,symbol,timestamp,close
1051372,GOOG,2024-11-20 22:37:00+00:00,5.178971
1051373,GOOG,2024-11-20 22:41:00+00:00,5.179534
1051374,GOOG,2024-11-20 22:47:00+00:00,5.179478
1051375,GOOG,2024-11-20 22:56:00+00:00,5.178463
1051376,GOOG,2024-11-21 00:00:00+00:00,5.180378


In [6]:
# Show timestamps in New York time so the session is 09:30–16:00 year-round (handles DST)
data["timestamp"] = data["timestamp"].dt.tz_convert("America/New_York")

prices = (
    data.pivot(index="timestamp", columns="symbol", values="close")
        .between_time("09:30", "15:59") # Normalizes time to regular business hours
        .dropna()
)
prices.head()
prices["GOOG"].shape

(372890,)

- score = more negative means more likely to be not cointegrated
- pvalue = Smaller value rejects null hypothesis of lack of cointegration
- crit_value = depends if 0, 1, or 2. Rejects cointegration at 99% probability if score < crit_value[0], 95% if score < crit_value[1], 90% if score < crit_value[2]

In [8]:
score, pvalue, crit_value = coint( 
    prices["GOOG"], 
    prices["GOOGL"],
    trend="c",
    autolag="AIC"
)

print(f"Score: {score}")
print(f"P-value: {pvalue}")
print(f"Critical Value {crit_value}")

Score: -4.694303397398274
P-value: 0.0005767728268452923
Critical Value [-3.89646937 -3.33614639 -3.04446137]


In [ ]:
if score < crit_value[0]:
    print("There is a 99% chance of cointegration!")
elif score < crit_value[1]:
    print("There is a 95% chance of cointegration!")
elif score < crit_value[2]:
    print("There is a 90% chance of cointegration!")
else:
    print("There is no statistically significant cointegration in the dataset. ")

There is no statistically significant cointegration in the dataset. 


Conclusion: NVDA and AMD aren't cointegrated, and thus can't be used for pairs trading